<style>
  .jp-RenderedMarkdown table { font-size: 1.0em; }
</style>
style cell

# Palmer  Penguin Unsupervised Machine Learning

## 1. 0 Dataset Selection and Description

### 1.1 Dataset Overview

The Palmer Penguins dataset is a popular dataset in the data science community, often used for classification tasks. The data was collected by Dr. Kirsten Gorman and the Palmer Station LTER Program from '2007–2009' to study ecological responses to environmental change. The data was packaged and released in July 2020, with data available undera CC0 license. The data set contains measurements of penguins from three different species: Adelie, Chinstrap, and Gentoo and includes features such as bill length, bill depth, flipper length, and  body mass, along with the species and island where the penguins were observed. The target variable for classification is the sex of the penguin, making it an ideal dataset for supervised machine learning classification tasks. The dataset is relevant for exploring how different physical characteristics can be used to classify penguin sex, which can have implications for ecological studies and conservation efforts.

### 1.2  Data Description

The dataset contains the following features:
- `species`: The species of the penguin (Adelie, Chinstrap, or Gentoo)
- `island`: The island where the penguin was observed (Biscoe, Dream, or Torgersen)
- `bill_length_mm`: The length of the penguin's bill in millimeters
- `bill_depth_mm`: The depth of the penguin's bill in millimeters
- `flipper_length_mm`: The length of the penguin's flipper in millimeters
- `body_mass_g`: The mass of the penguin in grams
- `sex`: The sex of the penguin (male or female)

### 1.3 Dataset Characteristics and Data Quality


| Attribute | Value                                             |
|-----------|---------------------------------------------------|
| Total Samples | 344                                               |
| Features (Inputs) | 4 numeric, continuous                             |
| Target Classes (Output) | 2 (sex)                                           |
| Samples per Class | 168 (Male), 165 (Female) (well balanced) |
| Missing Values | 11 (See Missing Value Report)                     |
| Data Type | Multivariate, real-valued                         |
| Task Type | Multi-class classification                        |
| Year Introduced | 2020 (Gorman)                                     |
| License | CC0                                               |




### 1.4 Features (Input Variables)

Each record represents measurements taken from a single penguin. All four features are **continuous numeric measurements** with body_mass recorded in grams and other features in millimeters:

| # | Feature Name         | Description | Unit | Range (approx.) |
|---|----------------------|-------------|------|-----------------|
| 1 | `bill_length`        | Length of the dorsal ridge of a penguin bill | mm   | 32.1 – 59.6     |
| 2 | `bill_depth`         | Depth of a penguin bill  | mm   | 13.1 – 21.5     |
| 3 | `flipper_length`   |Length of a penguin flipper | mm   | 172.0 – 231.0   |
| 4 | `body_mass`      | Weight of a penguin's body | g | 2700 – 6300  |


### 1.5 Target Class Variable

The target variable is the **sex** of the Palmer Penguins. There are exactly two sexes, represented by 168 Male and 165 Female samples:

| Class Label |  Sex   |  Count |
|--------------------|----------|-------|
| 0                  | *Male*   |   168 |
| 1                  | *Female* |  165 |

> 💡 **Key Observation:** The class distribution is perfectly balanced , which means **accuracy is a valid primary metric** and no class imbalance correction is needed for this dataset.

### 1.6 Relevance to the Classification Problem

The Palmer Penguins dataset is ideal for demonstrating supervised classification for several reasons:

- **Clarity:** The problem is well-defined — assign a species label to a new flower based on its measurements.
- **Tractability:** The small size (344 records rows) allows rapid prototyping and full hyperparameter search.
- **Minimal preprocessing burden:** Only 11 (3.20%)  missing values, no text encoding, and consistent units — allowing focus on modeling concepts.
- **Benchmark value:** Established ground truth makes it easy to validate whether an implementation is correct.

In a real-world context, this type of problem is analogous to **gender identification** in ecological surveys, **patient subtype classification** in medicine.


## 2. Analytical Approach and Model Development

### 2.1 Environment Setup

All code is written in **Python 3.11**. Install required packages once with:

```bash
pip install scikit-learn pandas numpy matplotlib seaborn
```


### 2.2 Import the Dataset


In [ ]:
# imports and standard

# ── Standard library & data ────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
from unicodedata import numeric
import sklearn as sk

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

# ── Dataset ────────────────────────────────────────────────────────────────
#from seaborn load_dataset
df = sns.load_dataset("penguins")

print(f'Shape:   {df.shape}')
print()
df.head()

### 2.3 Descriptive Statistics and Data Cleaning


`.describe()` reports count, mean, std, min, quartiles, and max for every numeric feature.
- **Input:** the four feature columns of `df`
- **Output:** a 8 × 4 summary statistics table

#### 2.3.1 Class Distribution Check

In [ ]:
df['sex'].value_counts()

#### 2.3.2 Column Summary

In [ ]:
# Create a summary table
#df[['species', 'island', 'sex']] = df[['species', 'island', 'sex']].astype(pd.StringDtype())
df = df. convert_dtypes()
info_df = pd.DataFrame({
    'Column': df.columns,
    'Non-Null Count': df.count().values,
    'Dtype': df.dtypes.values,
    'Null Count': df.isnull().sum().values
})
print("===Data Shape ===")
print(df.shape)
print("===Data Summary ===")
# Display the summary

info_df

### 2.4 Missing Value Analysis and Handling

Missing values must be detected before modeling. Real-world datasets often require imputation (median, mode, KNN) or row removal. The Palmer Penguin  dataset has eleven missing values for target 'sex'. The numerical columns (bill_length, bill_depth, flipper_length, and body_mass) each have two missing values, all of which are for the same record and these records are also missing the 'sex' category; thus there are a total of 11 records with missing values.  Those records represent 3.20% of the entire dataset and will be dropped from the analysis.

| Attribute                                     | Value                                |
|-----------------------------------------------|--------------------------------------|
| Original dataset shape                        | (344,7)                              |
| Cleaned dataset shape                         | (333,7)                              |
| Number of Records Dropped                     | 11                                   |
| Percentage of records with missing values     | 3.1977%                              |
| Percentage of records remaining | 96.8023% |


In [ ]:
# Input:  full DataFrame
# Output: count of NaN per column (0 = no missing data)
missing = df.isnull().sum()

missing_df = pd.DataFrame(missing, columns=['Missing Values'])
missing_df.index.name = 'Column'
missing_df

### 2.5 Exploratory Data Analysis (EDA)

#### 2.5.1 Pairplot — Feature Relationships by Species

A **pairplot** produces a grid of scatter plots for every pair of features, with KDE (density) plots on the diagonal. It visualizes the pairwise relationship between features, combining univariate analysis (distributions) and bivariate analysis (relationships) into one comprehensive view, enhancing rapid identification of patterns, correlations, and anomalies.

Coloring by species immediately reveals class separation.

- **Input:** DataFrame with 4 feature columns + `species_name` hue column
- **Output:** 4 × 4 figure; diagonal = KDE (univariate distribution per class)

In [ ]:
import matplotlib.pyplot as plt
df_cleaned = df.dropna()
penguins_df = df_cleaned.drop(columns=['sex'])
# Seaborn Pairplot
fig_seaborn_pairplot = sns.pairplot(penguins_df, hue='island', palette='viridis', height=2.5, aspect=1.2)
fig_seaborn_pairplot.fig.suptitle('Penguin Dataset — Feature Pairplot (Seaborn)', y=1.02) # Adjust title position
plt.show()

In [ ]:
#penguins_df = df_cleaned.drop(columns=['sex'])
penguins_df = df_cleaned

### 2.5.2 Violin Plot

The Violin Plot displays the distribution, probability density, and key summary statistics if of the data. The plot combines a box plot with a kernel density estimator, where the width of the "violin" represents the frequency of the data points at specific values.

In [ ]:
# Numeric columns for violin plots
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# Create a figure with subplots for each feature
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(numeric_cols):
    sns.violinplot(x='sex', y=feature, data=penguins_df, hue='sex', palette='viridis', ax=axes[i], inner='quartile', legend=False)
    axes[i].set_title(f'Penguins — {feature} Distribution by Sex (Violin Plot)')
    axes[i].set_xlabel('Sex')
    axes[i].set_ylabel(feature)

plt.tight_layout()
plt.show()

### 2.5.3 First few rows of the Cleaned Penguins Dataset

In [ ]:
penguins_df.head()

### 2.5.4 Feature Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
corr = penguins_df.iloc[:, 2:6].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    xticklabels=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'],
    yticklabels=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'],
    ax=ax
)
ax.set_title('Figure 3 — Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# print(corr.to_string())
# print()
# corr_table = corr.to_string
# print("")
# print("===Feature Correlation Matrix ===")
#
# corr

### 2.7 Classification Models

## 3.0 Model Selection and Justification

## 4. Key Findings and Insights

The model performance was similar and suggests that  all three models should be considered for further analysis and testing.

## 5. Limitations and Future Work

The dataset is small and thus, may cause overfitting, and the models may not generalize well to new data. The classes were well-balanced, and data quality was high, with few missing values for the initial dataset. The models would benefit from further tuning and validation on a larger dataset. Advanced classification techniques and models should also be considered for future work.